the result shows only 3 repos has instru testing signal in their POM.xmls

3 out of 4,518 is very low adoption rate and can be disregarded. 

it is not part of the analysis pipeline 

In [1]:
# -*- coding: utf-8 -*-
r"""
Scan downloaded pom.xml files for Android instrumentation-testing signals.

Input folder (downloaded POMs, named like owner.repo__maven++pom.xml):
    C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\All_POMs

Output folder:
    C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\All_POMs
Output file:
    pom_instrumentation_review.csv

Columns:
    file_name
    full_name           # owner.repo (from file name prefix before "__")
    is_android_maven    # android-maven-plugin/packaging=apk indication
    has_instrument_goal # plugin execution goal contains "instrument"
    has_runner_property # testInstrumentationRunner / instrumentationTestRunner present
    has_instrumentation_wording # generic "instrumentation" wording present
    packaging
    plugin_hits         # matched plugin coordinates found
    reasons             # short, human-readable rationale
    instru_in_pom       # final decision (True/False)
"""

import re
import csv
from pathlib import Path
from typing import List, Tuple, Optional
import xml.etree.ElementTree as ET

# ------------------ CONFIG ------------------
IN_DIR  = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\All_POMs")
OUT_DIR = Path(r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\All_POMs")
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_CSV = OUT_DIR / "pom_instrumentation_review.csv"

# Known Android Maven plugin coordinates (old & new groupIds)
ANDROID_MAVEN_PLUGINS: List[Tuple[str, str]] = [
    ("com.simpligility.maven.plugins", "android-maven-plugin"),
    ("com.jayway.maven.plugins.android.generation2", "android-maven-plugin"),
    # A few historical alternates sometimes seen:
    ("com.jayway.android.plugins", "android-maven-plugin"),
    ("org.jvending.maven.plugins", "android-maven-plugin"),
]

# Packaging values that typically indicate Android apps in Maven-era projects
ANDROID_PACKAGINGS = {"apk", "apklib", "aar"}

# Properties/element local-names that indicate instrumentation runner config
RUNNER_KEYS = {
    "testInstrumentationRunner",
    "instrumentationTestRunner",
    "android.test.instrumentationrunner",
    "android.instrumentationtestrunner",
}

# ------------------ Helpers ------------------
def ts():
    import datetime as _dt
    return _dt.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

def note(msg: str):
    print(f"[{ts()}] {msg}", flush=True)

def is_xml(text: str) -> bool:
    return text.lstrip().startswith("<")

def localname(tag: str) -> str:
    # Strip any XML namespace
    if "}" in tag:
        return tag.rsplit("}", 1)[1]
    return tag

def iter_local(root: ET.Element, name: str):
    for el in root.iter():
        if localname(el.tag) == name:
            yield el

def find_texts(root: ET.Element, name: str) -> List[str]:
    return [ (el.text or "").strip() for el in iter_local(root, name) if (el.text or "").strip() ]

def contains_instrument_goal(root: ET.Element) -> bool:
    # Find <plugins>/<plugin>/<executions>/<execution>/<goals>/<goal> and look for 'instrument'
    for plugin in iter_local(root, "plugin"):
        for goals_parent in iter_local(plugin, "goals"):
            for goal in iter_local(goals_parent, "goal"):
                if "instrument" in (goal.text or "").lower():
                    return True
    # Also check profile plugins
    for prof in iter_local(root, "profile"):
        for plugin in iter_local(prof, "plugin"):
            for goals_parent in iter_local(plugin, "goals"):
                for goal in iter_local(goals_parent, "goal"):
                    if "instrument" in (goal.text or "").lower():
                        return True
    return False

def list_android_plugins(root: ET.Element) -> List[str]:
    hits = []
    for plugin in iter_local(root, "plugin"):
        g = next(iter_local(plugin, "groupId"), None)
        a = next(iter_local(plugin, "artifactId"), None)
        gid = (g.text or "").strip() if g is not None else ""
        aid = (a.text or "").strip() if a is not None else ""
        for G, A in ANDROID_MAVEN_PLUGINS:
            if gid == G and aid == A:
                hits.append(f"{gid}:{aid}")
                break
    # Also check profiles
    for prof in iter_local(root, "profile"):
        for plugin in iter_local(prof, "plugin"):
            g = next(iter_local(plugin, "groupId"), None)
            a = next(iter_local(plugin, "artifactId"), None)
            gid = (g.text or "").strip() if g is not None else ""
            aid = (a.text or "").strip() if a is not None else ""
            for G, A in ANDROID_MAVEN_PLUGINS:
                if gid == G and aid == A:
                    hits.append(f"{gid}:{aid}")
                    break
    # De-dup preserve order
    seen, out = set(), []
    for x in hits:
        if x not in seen:
            seen.add(x); out.append(x)
    return out

def has_runner_property(root: ET.Element, raw_text_lower: str) -> bool:
    # Look for properties keys that match typical runner names
    for prop in iter_local(root, "properties"):
        # Scan all children names and texts
        for child in prop:
            name = localname(child.tag).lower()
            if (name in {k.lower() for k in RUNNER_KEYS}) or "instrumentation" in name:
                return True
            val = (child.text or "").lower()
            if "instrumentationtestrunner" in val or "testinstrumentationrunner" in val:
                return True
    # Also check any <configuration> blocks in plugins
    for conf in iter_local(root, "configuration"):
        # Look at tag names and values
        for child in conf.iter():
            nm = localname(child.tag).lower()
            if "instrumentation" in nm or "testrunner" in nm:
                return True
            val = (child.text or "").lower()
            if "instrumentationtestrunner" in val or "testinstrumentationrunner" in val:
                return True
    # Fallback: raw text grep
    if re.search(r"\b(test)?instrumentationtest(runner)?\b", raw_text_lower):
        return True
    return False

def has_generic_instrumentation_wording(raw_text_lower: str) -> bool:
    # Generic wording within a POM (avoid counting Gradle snippets)
    return "instrumentation" in raw_text_lower

def detect_packaging(root: ET.Element) -> str:
    vals = find_texts(root, "packaging")
    return vals[0].lower() if vals else ""

def analyze_pom(pom_text: str) -> dict:
    raw_lower = pom_text.lower()
    parsed = None
    try:
        parsed = ET.fromstring(pom_text)
    except Exception:
        parsed = None

    packaging = ""
    plugin_hits: List[str] = []
    has_goal = False
    has_runner = False

    if parsed is not None:
        packaging = detect_packaging(parsed)
        plugin_hits = list_android_plugins(parsed)
        has_goal = contains_instrument_goal(parsed)
        has_runner = has_runner_property(parsed, raw_lower)

    is_android_maven = bool(plugin_hits) or (packaging in ANDROID_PACKAGINGS) or ("android-maven-plugin" in raw_lower)
    has_instr_wording = has_generic_instrumentation_wording(raw_lower)

    # Final decision: "True" if we have explicit runner or explicit instrumentation goal,
    # or if it's an Android Maven project AND there is generic instrumentation wording
    # (weaker, but still a signal)
    instru_in_pom = bool(has_goal or has_runner or (is_android_maven and has_instr_wording))

    reasons = []
    if plugin_hits:
        reasons.append(f"plugins={';'.join(plugin_hits)}")
    if packaging:
        reasons.append(f"packaging={packaging}")
    if has_goal:
        reasons.append("goal:instrument*")
    if has_runner:
        reasons.append("runner:instrumentation")
    if has_instr_wording and not (has_goal or has_runner):
        reasons.append("mentions 'instrumentation'")

    return {
        "is_android_maven": bool(is_android_maven),
        "has_instrument_goal": bool(has_goal),
        "has_runner_property": bool(has_runner),
        "has_instrumentation_wording": bool(has_instr_wording),
        "packaging": packaging,
        "plugin_hits": ";".join(plugin_hits),
        "reasons": "; ".join(reasons),
        "instru_in_pom": bool(instru_in_pom),
    }

# ------------------ Main ------------------
def main():
    if not IN_DIR.exists():
        raise FileNotFoundError(f"Input directory not found: {IN_DIR}")

    rows = []
    files = sorted([p for p in IN_DIR.iterdir() if p.is_file() and p.suffix.lower() == ".xml"])
    note(f"Scanning {len(files)} pom.xml files from {IN_DIR}")

    for f in files:
        file_name = f.name
        full_name = file_name.split("__", 1)[0] if "__" in file_name else f.stem
        try:
            text = f.read_text(encoding="utf-8", errors="ignore")
        except Exception:
            text = ""

        result = analyze_pom(text)
        rows.append({
            "file_name": file_name,
            "full_name": full_name.lower(),
            "is_android_maven": result["is_android_maven"],
            "has_instrument_goal": result["has_instrument_goal"],
            "has_runner_property": result["has_runner_property"],
            "has_instrumentation_wording": result["has_instrumentation_wording"],
            "packaging": result["packaging"],
            "plugin_hits": result["plugin_hits"],
            "reasons": result["reasons"],
            "instru_in_pom": result["instru_in_pom"],
        })

    # Write CSV
    with OUT_CSV.open("w", newline="", encoding="utf-8-sig") as fp:
        writer = csv.DictWriter(fp, fieldnames=list(rows[0].keys()) if rows else [
            "file_name","full_name","is_android_maven","has_instrument_goal","has_runner_property",
            "has_instrumentation_wording","packaging","plugin_hits","reasons","instru_in_pom"
        ])
        writer.writeheader()
        for r in rows:
            writer.writerow(r)

    # Quick summary
    total = len(rows)
    flagged = sum(1 for r in rows if r["instru_in_pom"])
    note(f"Done. {flagged}/{total} POMs show instrumentation-testing signals.")
    note(f"Saved -> {OUT_CSV}")

if __name__ == "__main__":
    main()


[2025-08-24 09:39:21] Scanning 1380 pom.xml files from C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\All_POMs
[2025-08-24 09:39:21] Done. 24/1380 POMs show instrumentation-testing signals.
[2025-08-24 09:39:21] Saved -> C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\All_POMs\pom_instrumentation_review.csv
